# 🤖 Huấn Luyện Học Tăng Cường (PPO 9D) Cho Robot Tự Cân Bằng

**Quy chuẩn 6 bước Sim-to-Real toàn diện:**
1. **Mô phỏng vật lý & Domain Randomization:** Ngẫu nhiên hóa tải trọng $m \in [0.34, 0.38]\text{ kg}$, bánh xe $D=44\text{ mm}$, xung encoder $CPR=1400$, ma sát sàn, sụt áp pin và nhiễu MPU6050.
2. **Không gian Quan sát 9D:** $[s_t, s_{t-1}, a_{t-1}]$ giúp AI tự nhận biết đang lên dốc hoặc chở nặng để tự bơm lực bù mà không cần cảm biến phụ.
3. **Không gian Hành động:** Continuous $a_t \in [-1.0, 1.0]$ quy đổi ra xung PWM $[-999, +999]$.
4. **Hàm thưởng Anti-Limit-Cycle:** Ghìm chặt góc ($w_\theta=35.0$), cản góc mạnh ($w_{d\theta}=2.0$), cản vận tốc mạnh ($w_v=1.0$), triệt tiêu dao động vung lắc.
5. **Huấn luyện Stable-Baselines3 (PPO):** Mạng nơ-ron 2 tầng ẩn `net_arch=[64, 64]` với kích hoạt Tanh chuẩn xác.
6. **Xuất mô hình C Header:** Tự động sinh file `rl_sb3_stm32.h` với sai số SB3 $\equiv$ Numpy $\equiv$ C $\le 10^{-4}$ sẵn sàng nạp lên STM32L476RG.

--- 
### Bước 0: Cập nhật code mới nhất từ GitHub & Cài thư viện

In [ ]:
# Cập nhật repo nếu đã clone trước đó
import os
if not os.path.exists('configs'):
    !git clone -b p1-balance-robot https://github.com/nguyenthanhsang2306/self_blancing.git
    %cd self_blancing
else:
    !git fetch origin
    !git reset --hard origin/p1-balance-robot

# Cài đặt các thư viện RL cần thiết
!pip install gymnasium stable-baselines3 torch matplotlib pyyaml tensorboard

--- 
### Bước 1: Kiểm tra Môi trường 9D & Domain Randomization

In [ ]:
import yaml
from src.environment import SelfBalancingGymEnv

with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

env = SelfBalancingGymEnv(config)
obs, info = env.reset(seed=42)
print("✅ Observation Space:", env.observation_space.shape, "(Chuẩn 9D [s_t, s_{t-1}, a_{t-1}])")
print("✅ Action Space:     ", env.action_space)
print("✅ Giá trị quan sát ban đầu:", obs)

--- 
### Bước 2: Tiến hành Huấn luyện PPO (300,000 Timesteps)

In [ ]:
!python train.py --config configs/default.yaml --timesteps 600000

--- 
### Bước 3: Kiểm tra Đồng nhất Toán học (PyTorch <-> Numpy <-> C Header)

In [ ]:
!python verify_export.py

--- 
### Bước 4: Chạy Mô phỏng Kiểm thử Chống Va Đập Ngoại Lực

In [ ]:
!python inference.py

from IPython.display import Image, display
display(Image(filename='outputs/figures/rl_inference_test.png'))

--- 
### Bước 5: Tải file C Header `rl_sb3_stm32.h` về máy tính để nạp STM32

> **Lưu ý:** Nếu trình duyệt chặn Pop-up tự động tải về, bạn hãy mở menu cột trái của Colab (biểu tượng thư mục 📁) → vào `outputs/firmware_headers/rl_sb3_stm32.h` → Nhấp chuột phải chọn **Download**.

In [ ]:
# Đóng gói toàn bộ kết quả và header vào file zip
!zip -r rl_balancing_results.zip outputs/

from google.colab import files
try:
    print("⬇️ Đang kích hoạt tải file C Header về...")
    files.download('outputs/firmware_headers/rl_sb3_stm32.h')
    files.download('rl_balancing_results.zip')
except Exception as e:
    print(f"Tự động tải gặp giới hạn Pop-up ({e}).")
    print("📁 Bạn chỉ cần vào cột trái Colab -> outputs/firmware_headers/rl_sb3_stm32.h -> Chuột phải chọn Download là xong!")